# More Than a Hack 2027 — Xcelerate | Notebook complet Kaggle

**Objectif :** prédire pour chaque frame du match FC Barcelona – Athletic Club une classe :
- `H` = FC Barcelona en possession, ballon en jeu ; `A` = Athletic Club en possession, ballon en jeu ;
- `D` = ballon hors jeu ; `X` = abstention / frame non évaluée.

** Il se connecte au dossier Kaggle contenant tes fichiers ATD et Smart Tagging. La **vidéo n'est pas nécessaire** aux modèles présentés ici.

**Plan** : M1 = lire et aligner les données → M2 = reconstruire les features/baselines de l'équipe → M3 = exploiter les événements Smart Tagging correctement → M4 = modélisation temporelle avec plusieurs approches → contrôles, visualisation et CSV prêts à soumettre.

> **Confidentialité :** le dataset du match est soumis aux conditions de la compétition. **Garde le dataset et ce notebook Kaggle privés**, n'active pas « Publish » pour des données ou sorties qui les révèlent, et ne transmets ni vidéos ni extraits privés à un LLM hébergé. Vérifie que le stockage sur Kaggle est autorisé par tes conditions d'accès. Les fichiers CSV à transmettre officiellement doivent respecter la procédure de la compétition.

## Ce qu'il faut avoir sur Kaggle

les fichiers suivants dans ton dataset `BAR-DATA` :

- `BAR-ATH_tactical_atd.txt` : une ligne par image, joueurs + ballon ;
- `BAR-ATH_tactical_atd.xml` : correspondance des 46 slots de tracking avec les équipes ;
- `BAR-ATH_tactical_smart_tagging.xml` : événements tactiques automatiques ;
- `BAR-ATH_tactical_homography.csv` : matrices terrain → pixels ;
- deux fichiers `broadcast` (facultatifs et **non synchronisés**, donc non utilisés ici).

**Aucune vidéo n'est requise pour cet algorithme.** Si elle manque de Kaggle, ce n'est pas une erreur. La homographie sera vérifiée, mais elle n'est pas nécessaire tant qu'on n'exploite pas la vidéo.

Les fichiers d'entrée sont lus **en lecture seule** dans `/kaggle/input` ; les exports sont placés dans `/kaggle/working`.

In [1]:
# 0 — Imports et configuration : bibliothèque Kaggle standard uniquement.
import os
import sys
import math
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FPS = 25
EXPECTED_N = 199_319
FIELD_LENGTH_M = 105.3
FIELD_WIDTH_M = 68.0
LABELS = np.array(["H", "A", "D"], dtype="<U1")
# Intervalles publics publiés dans DATA.md (indices 0-based de la vidéo tactique).
HALVES = [(11_185, 90_651), (110_808, 190_197)]
SECOND_KICKOFF = 113_742   # les frames 110808..113741 de la période évaluée sont D

OUTPUT_DIR = Path(os.environ.get("MTAH_OUTPUT_DIR", "/kaggle/working"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# HSMM est optionnel car plus coûteux; HMM, heuristique et vote majoritaire tournent toujours.
RUN_HSMM = True
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__, "| Pandas:", pd.__version__)
print("Répertoire de sortie:", OUTPUT_DIR)

Python: 3.13.18 | NumPy: 2.1.3 | Pandas: 2.3.3
Répertoire de sortie: /kaggle/working


## M1 — 1. Trouver les fichiers Kaggle et vérifier les chemins

Ton chemin d'exemple est `/kaggle/input/datasets/erickambire/bar-data/...`.
Selon la façon dont Kaggle monte le dataset, il peut aussi apparaître sous `/kaggle/input/bar-data/...`.

Le code teste ces chemins, puis effectue une recherche de secours : **pas besoin de modifier les chemins à la main**, sauf si les fichiers portent d'autres noms.

In [4]:
# M1 — Résoudre le chemin du dataset sans dépendance à Colab ni à gdown.
def find_data_dir():
    candidates = [
        os.environ.get("MTAH_DATA_DIR", ""),
        "/kaggle/input/datasets/erickambire/bar-data",
        "/kaggle/input/bar-data",
    ]
    required = "BAR-ATH_tactical_atd.txt"
    for location in candidates:
        if location and (Path(location) / required).is_file():
            return Path(location)
    root = Path("/kaggle/input")
    if root.is_dir():
        hits = list(root.rglob(required))
        if len(hits) == 1:
            return hits[0].parent
        if len(hits) > 1:
            raise RuntimeError(f"Plusieurs datasets trouvés : {hits}. Définir MTAH_DATA_DIR.")
    raise FileNotFoundError(
        "ATD introuvable. Attache BAR-DATA dans Add Input à droite de Kaggle, "
        "ou définis la variable MTAH_DATA_DIR vers le dossier où se trouve l'ATD."
    )

DATA_DIR = find_data_dir()
PATHS = {
    "tracking": DATA_DIR / "BAR-ATH_tactical_atd.txt",
    "metadata": DATA_DIR / "BAR-ATH_tactical_atd.xml",
    "tagging": DATA_DIR / "BAR-ATH_tactical_smart_tagging.xml",
    "homography": DATA_DIR / "BAR-ATH_tactical_homography.csv",
}
for k, p in PATHS.items():
    print(f"{k:12s} | {'OK' if p.is_file() else 'ABSENT'} | {p.name}")
assert all(PATHS[k].is_file() for k in ("tracking", "metadata", "tagging")), "3 fichiers tactiques indispensables manquants"
print("Dossier détecté :", DATA_DIR)

tracking     | OK | BAR-ATH_tactical_atd.txt
metadata     | OK | BAR-ATH_tactical_atd.xml
tagging      | OK | BAR-ATH_tactical_smart_tagging.xml
homography   | OK | BAR-ATH_tactical_homography.csv
Dossier détecté : /kaggle/input/datasets/erickambire/bar-data


## M1 — 2. Lire le `.xml` des équipes

**Pourquoi ?** Le `.txt` stocke `Track 1`, `Track 2`, etc., mais pas le nom de l'équipe à chaque ligne. Le `.xml` précise à quelle équipe appartient chaque **slot de tracking**.

Attention : un slot n'est **pas** l'identité permanente d'un joueur. Lorsqu'une trajectoire se perd puis revient, un nouveau slot peut être créé. Les 46 slots ne sont donc pas 46 joueurs simultanés.

In [5]:
# M1 — Lecture XML avec ou sans namespace (plus robuste que des recherches XML exactes).
def local_name(tag):
    return tag.rsplit("}", 1)[-1].split(":")[-1]

def child_text(element, wanted):
    for child in element:
        if local_name(child.tag) == wanted:
            return (child.text or "").strip()
    return ""

def read_teams_and_slots(path):
    root = ET.parse(path).getroot()
    teams = [e for e in root.iter() if local_name(e.tag) == "Team"]
    players = [e for e in root.iter() if local_name(e.tag) == "Player"]
    if len(teams) != 2 or not players:
        raise ValueError(f"Métadonnées inattendues: {len(teams)} équipes, {len(players)} slots")
    names = {e.attrib["id"]: child_text(e, "Name") for e in teams}
    home_id, away_id = teams[0].attrib["id"], teams[1].attrib["id"]
    team_of_slot = np.array(
        [0 if e.attrib["teamId"] == home_id else 1 if e.attrib["teamId"] == away_id else -1 for e in players],
        dtype=np.int8
    )
    if (team_of_slot == -1).any():
        raise ValueError("Au moins un track slot n'appartient à aucune équipe connue")
    return names[home_id], names[away_id], team_of_slot

HOME_NAME, AWAY_NAME, TEAM_OF_SLOT = read_teams_and_slots(PATHS["metadata"])
N_SLOTS = len(TEAM_OF_SLOT)
print(f"H = {HOME_NAME} | A = {AWAY_NAME} | {N_SLOTS} slots")
print(f"Slots H: {(TEAM_OF_SLOT == 0).sum()} ; slots A: {(TEAM_OF_SLOT == 1).sum()}")

H = Barcelona | A = Atletic Club | 46 slots
Slots H: 24 ; slots A: 22


## M1 — 3. Transformer le tracking `.txt` en matrices

Chaque ligne de l'ATD a la forme : `frame_id:joueur1;joueur2;...:ballon`.
Chaque groupe joueur / ballon a trois valeurs `(x, y, vitesse)`, souvent `NaN` lorsqu'une observation manque.

Pour éviter le très gros tableau de dictionnaires du notebook Colab, on charge directement :

- `players.shape = (199319, 46, 3)` pour joueurs × coordonnées/vitesse ;
- `ball.shape = (199319, 3)` ;
- `frame_ids.shape = (199319,)`.

**Alignement obligatoire :** `frame_vidéo = frame_id_ATD - 1`.

In [6]:
# M1 — Parser vectorisé par ligne, mémoire prévisible ; aucune dépendance à BeautifulSoup.
def load_raw_atd(path, n_frames, n_slots):
    players = np.full((n_frames, n_slots, 3), np.nan, dtype=np.float32)
    ball = np.full((n_frames, 3), np.nan, dtype=np.float32)
    frame_ids = np.empty(n_frames, dtype=np.int32)
    row_count = 0
    t0 = time.perf_counter()
    with open(path, "r", encoding="utf-8") as f:
        for row_count, line in enumerate(f, start=1):
            if row_count > n_frames:
                raise ValueError(f"Plus de {n_frames} lignes dans le tracking")
            segments = line.strip().split(":", 2)
            if len(segments) != 3:
                raise ValueError(f"Ligne ATD mal formée {row_count}")
            f_id, players_txt, ball_txt = segments
            p = np.fromstring(players_txt.replace(";", ","), sep=",", dtype=np.float32)
            b = np.fromstring(ball_txt, sep=",", dtype=np.float32)
            if p.size != n_slots * 3 or b.size != 3:
                raise ValueError(f"Ligne {row_count}: trouvé {p.size} valeurs joueurs et {b.size} ballon")
            frame_ids[row_count - 1] = int(f_id)
            players[row_count - 1] = p.reshape(n_slots, 3)
            ball[row_count - 1] = b
    if row_count != n_frames:
        raise ValueError(f"Nombre de frames inattendu: {row_count} au lieu de {n_frames}")
    if not np.array_equal(frame_ids, np.arange(1, n_frames + 1)):
        raise ValueError("Frames ATD non consécutives 1..N ; vérifier l'ordre/source du fichier")
    print(f"ATD chargée : {row_count:,} frames, {n_slots} slots ; {time.perf_counter()-t0:.1f} s")
    return frame_ids, players, ball

frame_ids, players, ball = load_raw_atd(PATHS["tracking"], EXPECTED_N, N_SLOTS)
video_frames = frame_ids - 1
assert video_frames[0] == 0 and video_frames[-1] == EXPECTED_N - 1
ball_visible = np.isfinite(ball[:, 0]) & np.isfinite(ball[:, 1])
print(f"Ballon visible sur {ball_visible.mean():.1%} des frames totales")

ATD chargée : 199,319 frames, 46 slots ; 8.5 s
Ballon visible sur 56.9% des frames totales


## M1 — 4. Vérifier l'homographie (optionnelle)

L'homographie relie les coordonnées du terrain `(x, y)` aux pixels de la vidéo via une matrice 3×3. **Elle n'est pas nécessaire** pour un modèle fondé seulement sur le tracking et le Smart Tagging.

On vérifie donc l'alignement, sans allouer 199 319 matrices en mémoire.

In [7]:
# M1 — Vérification uniquement. N'exige pas de vidéo MP4.
if PATHS["homography"].is_file():
    h = pd.read_csv(PATHS["homography"], usecols=["frame", "valid", "reprojection_error"])
    assert len(h) == EXPECTED_N and np.array_equal(h["frame"].to_numpy(), video_frames)
    print("Homographie : OK, mêmes frames vidéo ; part valide :", f"{h['valid'].mean():.1%}")
else:
    print("Pas d'homographie : on continue (elle est optionnelle ici)")

Homographie : OK, mêmes frames vidéo ; part valide : 100.0%


## M2 — 5. Reconstruire les features numériques

Optimisée  M2 :

- nombre de joueurs détectés de chaque équipe ;
- vitesses moyenne et maximale ;
- distance **en mètres** du ballon au joueur le plus proche de chaque équipe ;
- nombre de joueurs à 2 m, 5 m et 10 m du ballon ;
- avantage de proximité et sa stabilité sur les cinq dernières frames.

Les axes sont normalisés 0–1 ; on multiplie `x` par **105,3 m** et `y` par **68 m** avant de calculer les distances.

**Correction :** quand le ballon est invisible, la distance et le nombre de joueurs proches sont **inconnus (`NaN`)**, et non égaux à zéro. Zéro voudrait dire « on sait qu'il n'y a personne près du ballon ».

In [8]:
# M2 — Reconstruire l'ensemble des caractéristiques de l'équipe, par matrices.
features = pd.DataFrame({"frame_id": frame_ids})  # alignement original M2 : ATD 1-based
features["video_frame"] = video_frames
features["ball_visible"] = ball_visible.astype(np.int8)
features["ball_x"] = ball[:, 0]
features["ball_y"] = ball[:, 1]
features["ball_speed"] = ball[:, 2]

for prefix, team_code in [("home", 0), ("away", 1)]:
    group = players[:, TEAM_OF_SLOT == team_code, :]
    visible = np.isfinite(group[:, :, 0]) & np.isfinite(group[:, :, 1])
    n_visible = visible.sum(axis=1)
    speed = np.where(visible & np.isfinite(group[:, :, 2]), group[:, :, 2], np.nan)
    count_speed = np.isfinite(speed).sum(axis=1)
    speed_sum = np.nansum(speed, axis=1)
    mean_speed = np.divide(speed_sum, count_speed, out=np.full(EXPECTED_N, np.nan), where=count_speed > 0)
    max_speed = np.where(np.isfinite(speed), speed, -np.inf).max(axis=1)
    max_speed[max_speed == -np.inf] = np.nan

    # Ces distances sont calculées *en mètres*.
    dx = (group[:, :, 0] - ball[:, None, 0]) * FIELD_LENGTH_M
    dy = (group[:, :, 1] - ball[:, None, 1]) * FIELD_WIDTH_M
    distance = np.sqrt(dx * dx + dy * dy)
    distance[~visible | ~ball_visible[:, None]] = np.inf
    nearest = distance.min(axis=1)
    nearest[~np.isfinite(nearest)] = np.nan

    features[f"{prefix}_players_visible"] = n_visible
    features[f"{prefix}_mean_speed"] = mean_speed
    features[f"{prefix}_max_speed"] = max_speed
    features[f"{prefix}_nearest_ball_dist"] = nearest
    for radius in [2, 5, 10]:
        count_near = (distance <= radius).sum(axis=1).astype(np.float32)
        count_near[~ball_visible] = np.nan
        features[f"{prefix}_players_within_{radius}m"] = count_near

features["nearest_dist_diff"] = features["home_nearest_ball_dist"] - features["away_nearest_ball_dist"]
features["home_closer"] = np.where(features["nearest_dist_diff"].isna(), np.nan,
                                   np.where(features["nearest_dist_diff"] < 0, 1.0,
                                            np.where(features["nearest_dist_diff"] > 0, 0.0, 0.5)))
features["nearest_dist_diff_change"] = features["nearest_dist_diff"].diff()
features.loc[0, "nearest_dist_diff_change"] = np.nan
features["home_closer_ratio_5"] = features["home_closer"].rolling(window=5, min_periods=3).mean()

# Exemple concret : une distance négative indique que Barcelona est plus proche.
display(features[["video_frame", "ball_visible", "home_nearest_ball_dist", "away_nearest_ball_dist",
                  "nearest_dist_diff", "home_closer_ratio_5"]].iloc[11_185:11_190].round(2))
print("Dimensions des features avant événements :", features.shape)

,video_frame,ball_visible,home_nearest_ball_dist,away_nearest_ball_dist,nearest_dist_diff,home_closer_ratio_5
11185,11185,0,NaN,NaN,NaN,NaN
11186,11186,0,NaN,NaN,NaN,NaN
11187,11187,1,8.87,0.42,8.45,NaN
11188,11188,1,9.54,1.12,8.41,NaN
11189,11189,1,10.21,1.84,8.37,0.0


Dimensions des features avant événements : (199319, 24)


## M2 — 6. Reproduire les deux baselines de l'équipe

Dans `Xcelerate_notebook_V2`, les dernières versions utilisent deux seuils de distance, **1 m** et **0,75 m**, avec la proportion de proximité sur cinq frames.

Exemple : `d_H = 1 m` et `d_A = 5 m` donnent `d_H - d_A = -4 m` → preuve en faveur de `H`, mais **pas une garantie de possession réelle** (rebond, passe, duel...).

`X` signifie ici « règle incapable de trancher » ; on ne doit **pas** le confondre avec `D` (« ballon hors jeu »).

In [9]:
# M2 — Baseline de M2, version symétrique finale des cellules 72–73.
def m2_baseline(df, threshold):
    ratio = df["home_closer_ratio_5"]
    diff = df["nearest_dist_diff"]
    visible = df["ball_visible"].eq(1)
    pred = np.full(len(df), "X", dtype="<U1")
    pred[(visible & diff.lt(-threshold) & ratio.gt(0.5)).to_numpy()] = "H"
    pred[(visible & diff.gt(threshold) & ratio.lt(0.5)).to_numpy()] = "A"
    return pred

features["baseline_prediction_1m"] = m2_baseline(features, 1.0)
features["baseline_prediction_075m"] = m2_baseline(features, 0.75)
print("Baseline M2 à 1 m :")
print(features["baseline_prediction_1m"].value_counts())
print("Aucun D : attendu, la baseline M2 ne modélise pas encore les arrêts de jeu.")

Baseline M2 à 1 m :
baseline_prediction_1m
X    105129
H     54203
A     39987
Name: count, dtype: int64
Aucun D : attendu, la baseline M2 ne modélise pas encore les arrêts de jeu.


## M3 — 7. Lire le Smart Tagging sans perdre les chevauchements

Le fichier XML contient **des intervalles** : par exemple `BUILD UP` de la seconde 600 à 610, avec l'équipe `Barcelona`.

L'ancien notebook construisait un dictionnaire `frame → un événement` : **si plusieurs annotations recouvraient la même frame, une seule restait**. Ici, on conserve **des compteurs d'événements simultanés**.

Nous ne considérons **pas** les événements automatiques comme une vérité terrain : un `BUILD UP` peut être décalé ou erroné, et une `SET PIECES` ne signifie pas `D` sur toute sa durée.

In [10]:
# M3 — Décodeur XML sans BeautifulSoup.
import unicodedata

def norm_text(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    return "".join(ch for ch in value if not unicodedata.combining(ch)).strip().lower()

def tag_to_side(value):
    v = norm_text(value)
    if "barcelona" in v:
        return "H"
    if "atletic" in v or "athletic" in v:
        return "A"
    return None   # N/A, inconnu

def read_smart_tagging(path, fps=FPS):
    root = ET.parse(path).getroot()
    events = []
    for inst in root.iter():
        if local_name(inst.tag).lower() != "instance":
            continue
        start = float(child_text(inst, "start"))
        end = float(child_text(inst, "end"))
        code = child_text(inst, "code").upper()
        tags = {}
        for lab in inst:
            if local_name(lab.tag) == "label":
                tags[child_text(lab, "group")] = child_text(lab, "text")
        events.append({
            "start_frame": int(round(start * fps)),
            "end_frame": int(round(end * fps)),
            "code": code, "side": tag_to_side(tags.get("Team")),
            "type": tags.get("Type", ""), "tags": tags,
        })
    return events

events = read_smart_tagging(PATHS["tagging"])
print("Annotations Smart Tagging :", len(events))
display(pd.Series([ev["code"] for ev in events]).value_counts().head(12).rename("nombre"))

Annotations Smart Tagging : 638


PLAYERS IN THE BOX              97
BALL IN FINAL THIRD             93
BALL IN THE BOX                 72
PROGRESSION                     66
DEFENDING IN MIDDLE THIRD       66
SET PIECES                      59
DEFENDING IN DEFENSIVE THIRD    30
CREATING CHANCES                30
LONG BALL                       29
DEFENSIVE TRANSITION            24
ATTACKING TRANSITION            24
BUILD UP                        22
Name: nombre, dtype: int64

## M3 — 8. Transformer les événements en signaux utilisables par M4

Exemple : si `BUILD UP` est marqué pour Barcelona entre les frames 15000 et 15250, `tag_attack_home` est incrémenté sur cette période. Si `DEFENDING...` concerne Athletic, ce n'est **pas automatiquement** une preuve que Barcelona contrôle le ballon : nous préférons ici conserver l'événement sans le transformer en label certain.

Nous créons trois types de signaux : indices offensifs `H/A`, activité des événements et présence de coups de pied arrêtés.

In [11]:
# M3 — Plusieurs événements peuvent être actifs à la même frame.
N = EXPECTED_N
m3_arrays = {name: np.zeros(N, dtype=np.int16) for name in [
    "tag_event_count", "tag_home_count", "tag_away_count",
    "tag_attack_home", "tag_attack_away", "tag_setpiece", "tag_setpiece_home", "tag_setpiece_away"
]}
ATTACKING_CODES = {
    "BUILD UP", "PROGRESSION", "CREATING CHANCES", "ATTACKING TRANSITION",
    "LONG BALL", "BALL IN FINAL THIRD", "BALL IN THE BOX", "PLAYERS IN THE BOX"
}
for ev in events:
    lo = max(0, ev["start_frame"])
    hi = min(N, ev["end_frame"] + 1)
    if hi <= lo:
        continue
    m3_arrays["tag_event_count"][lo:hi] += 1
    if ev["side"] == "H":
        m3_arrays["tag_home_count"][lo:hi] += 1
    elif ev["side"] == "A":
        m3_arrays["tag_away_count"][lo:hi] += 1
    if ev["code"] in ATTACKING_CODES:
        if ev["side"] == "H":
            m3_arrays["tag_attack_home"][lo:hi] += 1
        elif ev["side"] == "A":
            m3_arrays["tag_attack_away"][lo:hi] += 1
    if ev["code"] == "SET PIECES":
        m3_arrays["tag_setpiece"][lo:hi] += 1
        if ev["side"] == "H":
            m3_arrays["tag_setpiece_home"][lo:hi] += 1
        elif ev["side"] == "A":
            m3_arrays["tag_setpiece_away"][lo:hi] += 1

for name, values in m3_arrays.items():
    features[name] = values
print("Features M2 + M3 :", features.shape)
print("Frames avec au moins un événement :", (features["tag_event_count"] > 0).sum())
print("Frames avec au moins deux événements simultanés :", (features["tag_event_count"] > 1).sum())

Features M2 + M3 : (199319, 34)
Frames avec au moins un événement : 139665
Frames avec au moins deux événements simultanés : 95858


## Passage M1–M3 → M4 : contrat de données

Le fichier transmis à M4 (c'est à dire moi ) s'appelle `features_for_m4.csv`. Il contient les features numériques, les prédictions initiales de M2 et les compteurs tactiques de M3. Il contient **plus** de colonnes que la version Colab à 22 colonnes, parce que nous avons ajouté les informations tactiques et corrigé le traitement des valeurs manquantes.

Le fichier **n'est pas une vérité terrain**. Les prédictions `baseline_*` sont uniquement des propositions calculées par des règles.

In [12]:
# Export d'intégration pour M4, format compatible avec la version de l'équipe (séparateur ;).
FEATURES_PATH = OUTPUT_DIR / "features_for_m4.csv"
features.to_csv(FEATURES_PATH, sep=";", index=False, encoding="utf-8-sig")
assert len(features) == EXPECTED_N
assert not features.columns.duplicated().any()
print("Prêt pour M4 :", FEATURES_PATH)
print("Dimensions :", features.shape)
print("Colonnes :", features.columns.tolist())

Prêt pour M4 : /kaggle/working/features_for_m4.csv
Dimensions : (199319, 34)
Colonnes : ['frame_id', 'video_frame', 'ball_visible', 'ball_x', 'ball_y', 'ball_speed', 'home_players_visible', 'home_mean_speed', 'home_max_speed', 'home_nearest_ball_dist', 'home_players_within_2m', 'home_players_within_5m', 'home_players_within_10m', 'away_players_visible', 'away_mean_speed', 'away_max_speed', 'away_nearest_ball_dist', 'away_players_within_2m', 'away_players_within_5m', 'away_players_within_10m', 'nearest_dist_diff', 'home_closer', 'nearest_dist_diff_change', 'home_closer_ratio_5', 'baseline_prediction_1m', 'baseline_prediction_075m', 'tag_event_count', 'tag_home_count', 'tag_away_count', 'tag_attack_home', 'tag_attack_away', 'tag_setpiece', 'tag_setpiece_home', 'tag_setpiece_away']


# M4 — 9. Notre problème : les observations varient, la possession est continue

M2 peut prédire `H, H, A, H, H` parce que le ballon est momentanément plus près d'un défenseur adverse. Pourtant, une vraie possession peut être `H, H, H, H, H` si ce défenseur a seulement effleuré le ballon.

**Il faut distinguer deux niveaux :**

1. **Evidence / émission** : ce que disent les distances, les déplacements et les événements sur la frame actuelle ;
2. **Modèle temporel** : est-il plausible de changer d'équipe à cet instant, compte tenu de la continuité du jeu ?

Nous allons comparer des approches de complexité croissante, **sans ajuster les paramètres sur les 5 minutes de ground truth public**.

## M4 — 10. Construire des scores d'observation pour H, A et D

Les données ne contiennent **pas** directement `p_H`, `p_A`, `p_D` : c'est à nous de fabriquer un **modèle d'observation**.

Exemple fictif : `d_H=1 m`, `d_A=5 m` → distance favorable à `H`. En revanche, si le ballon disparaît, **ne jamais conclure automatiquement `D`**. On garde une information faible et on laisse la continuité temporelle décider.

La fonction ci-dessous construit des **scores heuristiques**, puis les transforme par softmax en trois nombres qui somment à 1. Ils ressemblent à des probabilités, mais **ne sont pas calibrés sur des labels** ; on ne doit pas les interpréter comme des probabilités fiables à 90 %.

**Limite connue :** la classe `D` est difficile sans détection précise de sortie de terrain / coup de sifflet. Nous incorporons des indices prudents de coups de pied arrêtés et d'immobilité, mais il faudra améliorer ce module avec M3.


# M4 — Construction du modèle d'observation

## 1. Quel problème cherchons-nous à résoudre ?

Notre objectif est de déterminer, à chaque instant d'un match de football, laquelle des trois situations suivantes est la plus probable :

- **H (Home)** : le FC Barcelone possède le ballon.
- **A (Away)** : l'Athletic Club possède le ballon.
- **D (Dead ball)** : le ballon est hors jeu.

Nous disposons déjà de caractéristiques (*features*) calculées par M1, M2 et M3 : positions des joueurs, distances au ballon, vitesses et événements tactiques.

Cependant, ces informations sont **des observations**, pas des réponses certaines.

**Exemple :**

À un instant donné :
- Un joueur de Barcelone se trouve à 1 m du ballon.
- Le joueur adverse le plus proche est à 5 m.
- Le Smart Tagging indique une phase offensive de Barcelone.

Ces observations suggèrent que Barcelone possède probablement le ballon. Mais elles ne le prouvent pas : il peut s'agir d'un ballon libre ou d'une interception en cours.

Nous voulons donc transformer plusieurs observations imparfaites en **trois scores**, correspondant à H, A et D.

C'est le rôle de la fonction `build_observation_scores()`.

---

## 2. Que reçoit la fonction en entrée ?

```python
def build_observation_scores(df):
```

`df` est un DataFrame Pandas : un tableau contenant les caractéristiques de chaque frame.

Par exemple :

| Frame | Ballon visible | Distance H | Distance A |
|---|---|---|---|
| 100 | True | 1 m | 5 m |
| 101 | True | 2 m | 4 m |
| 102 | False | NaN | NaN |

La fonction analysera toutes les lignes du tableau.

```python
n = len(df)
scores = np.zeros((n, 3), dtype=np.float64)
```

- `n` représente le nombre de frames.
- `np.zeros()` crée un tableau initialement rempli de zéros.
- Les trois colonnes correspondent respectivement à H, A et D.

Si nous avons 199 319 frames :

$$
\text{scores}\in\mathbb{R}^{199319\times3}
$$

Au départ, chaque frame possède donc trois scores nuls :

$$
S_t=[0,\ 0,\ 0]
$$

Nous allons progressivement modifier ces trois scores en fonction des indices disponibles.

**Important : à ce stade, ces nombres ne sont pas des probabilités.** Ce sont des scores numériques qui traduisent la force des différents indices.

---



## 3. Premier indice : la proximité du ballon

Le code commence par récupérer deux informations :

```python
diff = df["nearest_dist_diff"].to_numpy(dtype=float)
seen = df["ball_visible"].to_numpy(dtype=bool)
valid = seen & np.isfinite(diff)
```

### A. La différence de distance

Nous définissons :

$$
\Delta d=d_H-d_A
$$

Avec :
- $d_H$ : distance du joueur de Barcelone le plus proche du ballon.
- $d_A$ : distance du joueur d'Athletic le plus proche du ballon.

**Exemple :**

$$
d_H=1\,m,\qquad d_A=5\,m
$$

Alors :

$$
\Delta d=1-5=-4\,m
$$

La valeur est négative : Barcelone est plus proche du ballon.

Inversement, si $\Delta d>0$, Athletic est plus proche.

### B. Pourquoi `valid` ?

```python
valid = seen & np.isfinite(diff)
```

Nous devons vérifier deux conditions :

1. Le ballon est détecté (`seen = True`).
2. La différence de distance est un nombre valide (`np.isfinite`).

L'opérateur `&` signifie que les **deux conditions doivent être vraies**.

Si le ballon est invisible, nous ne devons pas utiliser une distance inconnue pour déterminer la possession.

### C. Convertir la distance en score

```python
prox = np.clip(
    np.nan_to_num(diff, nan=0.0) / 4.0,
    -2.0,
    2.0
)
```

Découpons cette opération :

- `np.nan_to_num()` remplace les valeurs manquantes `NaN` par 0.
- `/ 4.0` réduit l'amplitude des différences de distance.
- `np.clip(..., -2, 2)` limite la valeur finale à l'intervalle $[-2,2]$.

Pourquoi diviser par 4 ?

C'est une **hypothèse de conception** : une différence de distance de 4 mètres produit un score de proximité d'amplitude 1.

Pourquoi limiter entre −2 et 2 ?

Pour éviter qu'une très grande différence de distance domine automatiquement tous les autres indices.

Ces nombres ne sont pas des paramètres appris sur le match. Ils ont été choisis comme paramètres heuristiques et devront être justifiés.

### D. Ajouter l'indice aux équipes

```python
scores[:, 0] += np.where(valid, -prox, 0.0)
scores[:, 1] += np.where(valid, +prox, 0.0)
```

Explications :

- `scores[:, 0]` désigne la colonne H, pour toutes les frames.
- `scores[:, 1]` désigne la colonne A.
- `+=` signifie que nous ajoutons une valeur au score existant.
- `np.where(condition, a, b)` utilise `a` si la condition est vraie, sinon `b`.

Reprenons $\Delta d=-4$ :

$$
prox=\frac{-4}{4}=-1
$$

Le score de Barcelone augmente de $+1$, celui d'Athletic diminue de $-1$.

Nous obtenons :

$$
S=[1,-1,0]
$$

Le système favorise maintenant Barcelone, mais n'a pas encore pris en compte les autres informations.

---

## 4. Deuxième indice : les événements tactiques

Le Smart Tagging fournit des indications sur les phases offensives.

```python
scores[:, 0] += 0.40 * np.minimum(
    df["tag_attack_home"].to_numpy(), 3
)

scores[:, 1] += 0.40 * np.minimum(
    df["tag_attack_away"].to_numpy(), 3
)
```

`tag_attack_home` et `tag_attack_away` sont des caractéristiques construites lors de la préparation des données. Elles représentent des indices d'activité offensive pour chaque équipe.

`np.minimum(..., 3)` limite leur contribution à trois unités.

Le coefficient `0.40` définit le poids attribué à chaque unité d'indice tactique.

**Exemple :**

Supposons :
- `tag_attack_home = 1`
- `tag_attack_away = 0`

Le score de Barcelone augmente de :

$$
0.40\times1=0.40
$$

Nos scores deviennent :

$$
S=[1.40,-1,0]
$$

Pourquoi ne pas donner directement la possession à Barcelone ?

Parce que le Smart Tagging est généré automatiquement et peut contenir des erreurs. De plus, une phase offensive ne correspond pas nécessairement à une possession continue.

Nous utilisons donc cette information comme **un indice supplémentaire**, pas comme une vérité absolue.

---

## 5. Troisième indice : le ballon est-il hors jeu ?

Nous voulons maintenant construire le score de l'état D.

```python
scores[:, 2] -= 0.70
```

Nous commençons par soustraire 0.70.

C'est un biais initial : en l'absence d'autre information, notre système attribue un score plus faible à D qu'aux deux équipes.

Cette hypothèse ne signifie pas que le ballon est toujours en jeu.

### A. Les coups de pied arrêtés

```python
scores[:, 2] += 0.30 * np.minimum(
    df["tag_setpiece"].to_numpy(), 2
)
```

`tag_setpiece` indique la présence d'indices liés aux coups de pied arrêtés : corners, touches, coups francs, engagements, etc.

Ces événements peuvent être associés à une interruption du jeu.

Si `tag_setpiece = 1`, le score D augmente de 0.30.

Attention : un coup franc peut déjà avoir été joué. La présence d'un tel événement ne signifie donc pas que chaque frame de son intervalle est hors jeu.

### B. Ballon proche d'un joueur

```python
closest = np.minimum(
    np.nan_to_num(home_dist, nan=np.inf),
    np.nan_to_num(away_dist, nan=np.inf)
)

scores[:, 2] -= 1.2 * (
    valid & (closest <= 2.0)
)
```

Nous déterminons la distance entre le ballon et le joueur le plus proche, quelle que soit son équipe.

Si le ballon est visible, avec une distance inférieure ou égale à 2 mètres, nous réduisons le score D de 1.2.

Pourquoi ?

Parce qu'un ballon très proche d'un joueur est un indice possible de jeu actif.

Cependant, cette règle n'est pas absolue : un ballon placé au sol avant un coup franc peut également être proche d'un joueur.

### C. Ballon invisible et joueurs peu mobiles

```python
slow = (
    (home_speed < 1.2) &
    (away_speed < 1.2)
)

scores[:, 2] += 0.50 * ((~seen) & slow)
```

*Cette écriture simplifiée illustre les conditions ; le code original traite aussi les valeurs manquantes.*

Ici :
- `home_speed` et `away_speed` sont les vitesses moyennes des joueurs détectés dans les deux équipes.
- `~seen` signifie que le ballon n'est pas visible.
- `slow` vaut `True` lorsque les deux vitesses moyennes sont inférieures à 1,2 m/s.

Si le ballon est invisible **et** que les deux équipes bougent très lentement, nous augmentons le score D de 0.50.

Pourquoi ?

Cela pourrait correspondre à une interruption de jeu.

Mais le ballon peut aussi être invisible pendant une passe aérienne. C'est donc seulement un indice faible.

### D. Ballon détecté hors du terrain

```python
outside = seen & (
    (bx < -0.01) |
    (bx > 1.01) |
    (by < -0.01) |
    (by > 1.01)
)

scores[:, 2] += 2.0 * outside
```

Les coordonnées du terrain sont normalisées :

$$
0\leq x\leq1,\qquad 0\leq y\leq1
$$

Si les coordonnées du ballon dépassent légèrement les limites tolérées, le code considère cela comme un indice de sortie du terrain.

L'opérateur `|` signifie **OU**.

Le score D augmente alors de 2.0.

Les seuils −0.01 et 1.01 fournissent une petite marge autour du terrain.

Il reste cependant une limite : un ballon détecté hors du rectangle de référence ne prouve pas à lui seul que l'arbitre a arrêté le jeu, notamment si les données de tracking sont bruitées.

---



## 6. Transformer les scores en valeurs normalisées : Softmax

Nous avons accumulé plusieurs indices dans les trois scores :

$$
S=[S_H,S_A,S_D]
$$

La fonction :

```python
def softmax_rows(logits):
    shifted = logits - np.max(
        logits, axis=1, keepdims=True
    )
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
```

transforme ces scores en valeurs positives dont la somme vaut 1.

La formule mathématique est :

$$
e_k=\frac{\exp(S_k)}
{\exp(S_H)+\exp(S_A)+\exp(S_D)}
$$

où $k$ représente l'un des trois états.

La fonction exponentielle `exp()` transforme chaque score avant la normalisation.

Pourquoi soustraire le maximum ?

```python
shifted = logits - np.max(logits, axis=1, keepdims=True)
```

Cette opération améliore la stabilité numérique : elle évite de calculer l'exponentielle de valeurs très grandes.

Elle ne change pas le résultat mathématique du Softmax.

### Exemple numérique complet

Reprenons notre exemple :

- Différence de distance : −4 m.
- Un indice offensif pour Barcelone.
- Aucun indice offensif pour Athletic.
- Aucun set piece.
- Ballon visible et proche d'un joueur.
- Ballon non détecté hors terrain.

Nous obtenons :

$$
S_H=1+0.40=1.40
$$

$$
S_A=-1
$$

$$
S_D=-0.70-1.20=-1.90
$$

Donc :

$$
S=[1.40,-1,-1.90]
$$

Après Softmax, nous obtenons approximativement :

| État | Score initial | Valeur normalisée |
|---|---:|---:|
| H | 1.40 | 0.887 |
| A | −1.00 | 0.080 |
| D | −1.90 | 0.033 |

La somme est égale à 1, à l'arrondi près.

Le système favorise fortement Barcelona à cet instant.

**Attention : 0.887 n'est pas nécessairement une probabilité réelle de possession de 88,7 %.** C'est une valeur normalisée produite par nos règles heuristiques. Sans calibration sur des données indépendantes autorisées, elle ne garantit pas une fréquence de réussite de 88,7 %.

---

## 7. Quelle est la sortie finale de la fonction ?

```python
return softmax_rows(scores)
```

Cette instruction renvoie un tableau de dimension :

$$
E\in\mathbb{R}^{N\times3}
$$

Pour chaque frame $t$, nous avons :

$$
E_t=[e_H,e_A,e_D]
$$

Par exemple :

| Frame | e_H | e_A | e_D |
|---|---:|---:|---:|
| 100 | 0.887 | 0.080 | 0.033 |
| 101 | 0.650 | 0.280 | 0.070 |
| 102 | 0.350 | 0.600 | 0.050 |

Ces valeurs sont illustratives.

L'instruction :

```python
emissions = build_observation_scores(features)
```

calcule ces trois valeurs pour toutes les frames.

Puis :

```python
assert emissions.shape == (EXPECTED_N, 3)
```

vérifie que le tableau possède le nombre attendu de frames et exactement trois colonnes.

```python
assert np.allclose(emissions.sum(axis=1), 1.0)
```

vérifie que les trois valeurs de chaque frame totalisent approximativement 1.

Enfin :

```python
display(
    pd.DataFrame(
        emissions[11_185:11_190],
        columns=["e_H", "e_A", "e_D"]
    ).round(3)
)
```

affiche cinq lignes, correspondant aux indices 11185 à 11189, et arrondit les résultats à trois décimales.

---

## 8. Est-ce déjà notre Hidden Markov Model ?

**Non. Nous venons de construire le modèle d'observation, pas encore le HMM complet.**

À chaque frame, nous disposons seulement de trois scores.

Or, notre objectif est de reconstruire une **séquence de possessions cohérentes dans le temps**.

Imaginons les observations suivantes :

| Frame | e_H | e_A | e_D |
|---|---:|---:|---:|
| 100 | 0.90 | 0.07 | 0.03 |
| 101 | 0.85 | 0.12 | 0.03 |
| 102 | 0.35 | 0.60 | 0.05 |
| 103 | 0.88 | 0.09 | 0.03 |

Si nous sélectionnons simplement la plus grande valeur à chaque frame, nous obtenons :

$$
H\rightarrow H\rightarrow A\rightarrow H
$$

Mais le passage isolé vers Athletic peut être causé par une erreur de tracking ou un simple rebond.

**Le rôle du HMM sera de déterminer si ce changement est suffisamment crédible pour représenter une véritable récupération du ballon.**

Pour cela, nous ajouterons une matrice de probabilités de transition entre H, A et D, puis un algorithme de décodage temporel appelé **Viterbi**.

Dans le code actuel, les valeurs du Softmax seront utilisées comme *potentiels d'observation* du HMM. Ce ne sont pas des probabilités d'émission $P(O_t\mid Z_t)$ rigoureusement estimées ; leur utilisation est une approximation à garder à l'esprit.

---



## 9. Limites de notre modèle d'observation

Avant de poursuivre, il faut retenir quatre limites importantes.

1. **Les coefficients sont heuristiques.** Les valeurs 0.40, 0.70, 1.20 ou 2.0 ne proviennent pas d'un entraînement statistique. Elles représentent des hypothèses métier.

2. **La proximité ne garantit pas le contrôle.** Un joueur peut être le plus proche du ballon sans réellement en avoir la possession.

3. **L'état D reste difficile à détecter.** Un ballon invisible n'est pas nécessairement hors jeu, et les indices de faible vitesse ou de set piece restent imparfaits.

4. **Chaque frame est évaluée indépendamment.** Cette fonction ne vérifie pas encore si les changements de possession sont cohérents avec les frames précédentes et suivantes.

**Conclusion :**

Cette fonction constitue la première couche de notre système M4. Elle transforme les observations de M1, M2 et M3 en trois scores normalisés.

La prochaine couche, le HMM, exploitera ces scores et les transitions entre les états pour produire une séquence temporelle de possessions.


In [13]:
# M4 — Modèle d'observation explicable, paramétré par hypothèses métier (pas par ground truth).
def softmax_rows(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

def build_observation_scores(df):
    n = len(df)
    scores = np.zeros((n, 3), dtype=np.float64)  # colonnes H, A, D
    diff = df["nearest_dist_diff"].to_numpy(dtype=float)
    seen = df["ball_visible"].to_numpy(dtype=bool)
    valid = seen & np.isfinite(diff)

    # Avantage de proximité : négatif -> H, positif -> A.
    prox = np.clip(np.nan_to_num(diff, nan=0.0) / 4.0, -2.0, 2.0)
    scores[:, 0] += np.where(valid, -prox, 0.0)
    scores[:, 1] += np.where(valid, +prox, 0.0)

    # Indice tactique offensif : la relation « attaque » -> « possession » est indicative.
    scores[:, 0] += 0.40 * np.minimum(df["tag_attack_home"].to_numpy(), 3)
    scores[:, 1] += 0.40 * np.minimum(df["tag_attack_away"].to_numpy(), 3)

    # Les phases de set piece peuvent entourer un arrêt, mais ne sont PAS toutes "D".
    scores[:, 2] -= 0.70
    scores[:, 2] += 0.30 * np.minimum(df["tag_setpiece"].to_numpy(), 2)

    home_dist = df["home_nearest_ball_dist"].to_numpy(dtype=float)
    away_dist = df["away_nearest_ball_dist"].to_numpy(dtype=float)
    closest = np.minimum(np.nan_to_num(home_dist, nan=np.inf), np.nan_to_num(away_dist, nan=np.inf))
    # Ballon très proche d'un joueur -> probablement jeu actif (mais pas certain).
    scores[:, 2] -= 1.2 * (valid & (closest <= 2.0))

    # Ballon absent ET très faible mouvement collectif : indice faible d'arrêt possible.
    home_speed = df["home_mean_speed"].to_numpy(dtype=float)
    away_speed = df["away_mean_speed"].to_numpy(dtype=float)
    slow = (np.nan_to_num(home_speed, nan=100.0) < 1.2) & (np.nan_to_num(away_speed, nan=100.0) < 1.2)
    scores[:, 2] += 0.50 * ((~seen) & slow)

    # Ballon détecté nettement hors terrain : indice de D, à traiter avec prudence.
    bx = df["ball_x"].to_numpy(dtype=float)
    by = df["ball_y"].to_numpy(dtype=float)
    outside = seen & ((bx < -0.01) | (bx > 1.01) | (by < -0.01) | (by > 1.01))
    scores[:, 2] += 2.0 * outside
    return softmax_rows(scores)

emissions = build_observation_scores(features)
assert emissions.shape == (EXPECTED_N, 3)
assert np.allclose(emissions.sum(axis=1), 1.0)
print("Exemple de scores (non calibrés) :")
display(pd.DataFrame(emissions[11_185:11_190], columns=["e_H", "e_A", "e_D"]).round(3))

Exemple de scores (non calibrés) :


,e_H,e_A,e_D
0,0.231,0.514,0.255
1,0.231,0.514,0.255
2,0.008,0.980,0.012
3,0.008,0.980,0.012
4,0.008,0.980,0.012


## M4 — 11. Modèle A : vote majoritaire temporel

C'est le premier modèle de référence. On estime, quand le ballon est visible, l'équipe du joueur le plus proche. Puis on compte les votes `H` et `A` dans une fenêtre autour de chaque frame.

**Exemple :** dans `H H A H H`, le vote majoritaire tend vers `H H H H H`.

**Limites :** il peut supprimer une interception réelle mais courte ; il ne modélise pas réellement les arrêts de jeu. Nous reproduisons le **principe du baseline officiel**, sans prétendre reproduire ses chiffres privés si les données ou le découpage diffèrent.

In [14]:
# M4 — Vote majoritaire sur une fenêtre centrée ±window ; très rapide grâce aux cumuls.
def nearest_raw_codes(df):
    diff = df["nearest_dist_diff"].to_numpy(dtype=float)
    seen = df["ball_visible"].to_numpy(dtype=bool)
    raw = np.full(len(df), -1, dtype=np.int8)  # -1 = pas de preuve de joueur proche
    raw[seen & np.isfinite(diff) & (diff <= 0)] = 0
    raw[seen & np.isfinite(diff) & (diff > 0)] = 1
    return raw

def majority_vote(raw, half_width=50):
    n = len(raw)
    if half_width <= 0:
        return np.where(raw == 0, "H", np.where(raw == 1, "A", "D"))
    h = np.r_[0, np.cumsum(raw == 0)]
    a = np.r_[0, np.cumsum(raw == 1)]
    t = np.arange(n)
    left = np.maximum(t - half_width, 0)
    right = np.minimum(t + half_width + 1, n)
    n_h = h[right] - h[left]
    n_a = a[right] - a[left]
    pred = np.full(n, "D", dtype="<U1")
    mask = (n_h + n_a) > 0
    pred[mask] = np.where(n_h[mask] >= n_a[mask], "H", "A")
    return pred

raw = nearest_raw_codes(features)
models = {
    "nearest_raw": majority_vote(raw, half_width=0),
    "majority_50": majority_vote(raw, half_width=50),
}
print("Modèles de base :", list(models))

Modèles de base : ['nearest_raw', 'majority_50']


## M4 — 12. Modèle B : hystérésis (filtrage des changements trop brefs)

Ce modèle conserve la possession courante tant qu'une nouvelle classe n'est pas soutenue **plusieurs frames consécutives**.

Exemple : `H H A H H` → garde `H` si l'alerte `A` ne dure qu'une frame. Mais `H H A A A A` peut déclencher un changement vers `A`.

Avantage : simple, interprétable et fonctionne sans entraînement. Limite : retarde délibérément les transitions ; le `spellF1` est sensible à leur position.

In [15]:
# M4 — Filtre causal sur scores d'observation, remis à zéro à chaque mi-temps.
def decode_hysteresis(prob, blocks, persist_frames=10, switch_margin=0.10):
    labels = np.full(len(prob), "X", dtype="<U1")
    for start, stop in blocks:
        current = int(np.argmax(prob[start]))
        candidate, count = -1, 0
        for t in range(start, stop + 1):
            best = int(np.argmax(prob[t]))
            convincing = prob[t, best] >= prob[t, current] + switch_margin
            if best != current and convincing:
                count = count + 1 if best == candidate else 1
                candidate = best
                if count >= persist_frames:
                    current = best
                    candidate, count = -1, 0
            else:
                candidate, count = -1, 0
            labels[t] = LABELS[current]
    return labels

models["hysteresis"] = decode_hysteresis(emissions, HALVES)
print("Hystérésis prêt.")

Hystérésis prêt.


## M4 — 13. Modèle C : Hidden Markov Model (HMM) + Viterbi

Le **HMM** suppose que la véritable possession `S_t` (H/A/D) est un **état caché**. Nous observons seulement des indices : distance du ballon, mouvement des joueurs, événements.

Il combine :

- **Émission** : comment les observations sont compatibles avec H, A ou D ;
- **Transition** : à quel point il est plausible de rester dans le même état ou d'en changer.

La meilleure suite d'états est obtenue par l'algorithme de **Viterbi** :

\[
\delta_t(j)=\log e_t(j)+\max_i\left[\delta_{t-1}(i)+\log T_{ij}\right].
\]

Ci-dessous, les valeurs de `T` sont des **hypothèses structurelles de départ**, pas des statistiques apprises sur les cinq minutes publiques. Les lignes somment à 1. `T[H,H]` élevé rend les micro-changements plus coûteux.

Nous décodons chaque mi-temps indépendamment pour ne pas propager l'état à travers la pause.

In [16]:
# M4 — Matrice de transitions H/A/D et décodage Viterbi en log pour la stabilité numérique.
TRANSITION = np.array([
    [0.993, 0.003, 0.004],  # depuis H -> H,A,D
    [0.003, 0.993, 0.004],  # depuis A -> H,A,D
    [0.015, 0.015, 0.970],  # depuis D -> H,A,D
], dtype=np.float64)
assert np.allclose(TRANSITION.sum(axis=1), 1.0)

def viterbi_hmm(probs, blocks, transition=TRANSITION):
    result = np.full(len(probs), "X", dtype="<U1")
    log_t = np.log(np.maximum(transition, 1e-12))
    for lo, hi in blocks:
        obs = np.log(np.maximum(probs[lo:hi + 1], 1e-12))
        m = obs.shape[0]
        score = np.log(np.array([0.45, 0.45, 0.10])) + obs[0]
        back = np.empty((m, 3), dtype=np.uint8)
        back[0, :] = 0
        for t in range(1, m):
            cand = score[:, None] + log_t
            back[t] = np.argmax(cand, axis=0)
            score = obs[t] + cand[back[t], np.arange(3)]
        states = np.empty(m, dtype=np.uint8)
        states[-1] = np.argmax(score)
        for t in range(m - 1, 0, -1):
            states[t - 1] = back[t, states[t]]
        result[lo:hi + 1] = LABELS[states]
    return result

models["hmm_viterbi"] = viterbi_hmm(emissions, HALVES)
print("HMM Viterbi prêt.")

HMM Viterbi prêt.


## M4 — 14. Vérification du HMM sur un exemple synthétique (autorisé)

Nous créons une petite séquence artificielle avec 15 frames de possession H, puis 15 de possession A. Une observation ponctuelle est perturbée. Le but est simplement de vérifier que le programme est capable de faire disparaître une fausse transition, **sans regarder ni entraîner sur la vérité terrain du match**.

In [ ]:
# Test unitaire didactique, uniquement données inventées.
toy = np.array([[0.90, 0.08, 0.02]] * 15 + [[0.08, 0.90, 0.02]] * 15, dtype=float)
toy[7] = [0.05, 0.94, 0.01]  # une fausse alerte A au milieu de H
toy_pred = viterbi_hmm(toy, [(0, len(toy)-1)])
print("Observations brutes :", "".join(LABELS[np.argmax(toy, axis=1)]))
print("HMM décodé         :", "".join(toy_pred))
assert len(toy_pred) == 30 and set(toy_pred).issubset({"H", "A", "D"})

## M4 — 15. Modèle D (option) : HSMM — Hidden Semi-Markov Model

Le HMM suppose une durée **géométrique implicite** des possessions. Un **HSMM** ajoute explicitement une durée de segment, par exemple « une possession de 0,2 seconde est peu plausible ; une possession de plusieurs secondes est plus plausible ».

Pour limiter le coût de calcul, nous décodons sur une grille **de 5 Hz** (une observation toutes les 5 frames à partir des scores moyens), puis remontons à 25 Hz. **Limite :** une transition peut être décalée jusqu'à ~0,2 s. L'algorithme reste un vrai décodage par durée de segment, mais c'est une **approximation temporelle** de la résolution originale.

Nous utilisons une loi de durée **log-normale discrète fixée a priori** ; ce n'est ni une durée mesurée dans le ground truth ni une hyperparamétrisation obtenue en optimisant sur les cinq minutes publiques.

In [ ]:
# M4 — Décodage HSMM : programmation dynamique sur segments et durée explicite.
# À 5 Hz le match fait ~40 000 observations. Le modèle reste optionnel (RUN_HSMM).
def duration_log_prior(max_steps, median_seconds, fps_low, sigma=0.90):
    d = np.arange(1, max_steps + 1, dtype=float)
    sec = d / fps_low
    # Densité log-normale discrète (pénalise très courtes et très longues durées).
    z = (np.log(sec) - np.log(median_seconds)) / sigma
    logp = -0.5*z*z - np.log(sec) - np.log(sigma)  # constante commune omise
    logp -= np.log(np.exp(logp - logp.max()).sum()) + logp.max()
    return logp

def hsmm_one_block(prob, factor=5, max_duration_s=90, fps=FPS):
    '''Viterbi semi-markov explicite, états H/A/D, depuis observations moyennées par groupe.
    Résultat à la fréquence originale. La durée max est un compromis : longue possession > max
    sera découpée, ce qui est une limitation importante du modèle.
    '''
    n = len(prob)
    low_fps = fps / factor
    pad = (-n) % factor
    padded = np.pad(prob, ((0,pad),(0,0)), mode="edge")
    low = padded.reshape(-1, factor, 3).mean(axis=1)
    T = len(low)
    logs = np.log(np.maximum(low, 1e-12))
    prefix = np.vstack([np.zeros((1,3)), np.cumsum(logs, axis=0)])
    max_d = min(T, int(max_duration_s * low_fps))
    # Transitions entre segments distincts : probabilité de changement relative aux autres.
    log_switch = np.log(np.array([
        [1e-12, 0.42, 0.58],
        [0.42, 1e-12, 0.58],
        [0.50, 0.50, 1e-12],
    ], dtype=float))
    dur = np.column_stack([
        duration_log_prior(max_d, median_seconds=12, fps_low=low_fps),
        duration_log_prior(max_d, median_seconds=12, fps_low=low_fps),
        duration_log_prior(max_d, median_seconds=3.5, fps_low=low_fps),
    ])
    dp = np.full((T+1,3), -np.inf, dtype=np.float64)
    dp[0] = np.log([0.45,0.45,0.10])
    previous_time = np.zeros((T+1,3), dtype=np.int32)
    previous_state = np.zeros((T+1,3), dtype=np.uint8)
    for t in range(1,T+1):
        d = np.arange(1, min(t,max_d)+1)
        starts = t-d
        for s in range(3):
            e = prefix[t,s] - prefix[starts,s]
            prev = dp[starts,:] + log_switch[:,s][None,:]
            # Début d'une mi-temps : prior d'état initial, sans changement d'état.
            zero_start = (starts == 0)
            if np.any(zero_start):
                prev[zero_start,:] = -np.inf
                prev[zero_start,s] = dp[0,s]
            best_state = np.argmax(prev,axis=1)
            best_prev = prev[np.arange(len(d)),best_state]
            vals = best_prev + e + dur[d-1,s]
            j = int(np.argmax(vals))
            dp[t,s] = vals[j]
            previous_time[t,s] = starts[j]
            previous_state[t,s] = best_state[j]
    label_low = np.empty(T,dtype=np.uint8)
    t = T
    state = int(np.argmax(dp[T]))
    while t > 0:
        start = int(previous_time[t,state])
        label_low[start:t] = state
        state, t = int(previous_state[t,state]), start
    return LABELS[np.repeat(label_low,factor)[:n]]

def decode_hsmm(prob, blocks):
    out = np.full(len(prob),"X",dtype="<U1")
    for i,(start,end) in enumerate(blocks,1):
        print(f"HSMM : décodage mi-temps {i} ({end-start+1:,} frames)")
        out[start:end+1] = hsmm_one_block(prob[start:end+1])
    return out

if RUN_HSMM:
    t0=time.perf_counter()
    models["hsmm_5hz"] = decode_hsmm(emissions, HALVES)
    print(f"HSMM terminé en {time.perf_counter()-t0:.1f} s")
else:
    print("HSMM désactivé ; basculer RUN_HSMM = True pour l'essayer")

## M4 — 16. Post-traitement : respecter les zones évaluées

Les fichiers doivent avoir **199 319 lignes**, y compris les périodes non notées. Pour ne pas transférer une possession par erreur entre les mi-temps :

- avant le match, pendant la pause et après le match : `X` ;
- deuxième période, du début de l'intervalle évalué jusqu'au véritable engagement (110 808–113 741) : `D`, information explicitement **publiée** par les organisateurs ;
- sur le reste des deux mi-temps : prédictions de chaque modèle.

Il ne s'agit pas d'utiliser un ground truth privé, mais des **bornes temporelles rendues publiques**.

In [ ]:
# Validation du contrat temporel, sans fuite entre mi-temps.
def apply_published_boundaries(pred):
    pred = np.asarray(pred, dtype="<U1").copy()
    assert len(pred) == EXPECTED_N
    mask = np.zeros(EXPECTED_N,dtype=bool)
    for a,b in HALVES:
        mask[a:b+1]=True
    pred[~mask] = "X"
    pred[HALVES[1][0]:SECOND_KICKOFF] = "D"
    assert set(np.unique(pred)).issubset({"H","A","D","X"})
    return pred

models = {name:apply_published_boundaries(pred) for name,pred in models.items()}
print("Modèles :",list(models))

## M4 — 17. Comparer les modèles **sans** vérité terrain

Nous pouvons compter les classes, les transitions, la durée des possessions **prédites** et le nombre de courtes séquences. Ce sont des **diagnostics**, pas des scores de qualité.

**Attention :** moins de transitions ne signifie pas automatiquement un meilleur `spellF1`. Un modèle trop lisse effacera les véritables interceptions. Le `spellF1` officiel nécessite la vérité terrain complète, qui n'est pas disponible.

In [ ]:
# M4 — Compter les possessions H/A en interrompant les séquences par D ou X.
def possession_stats(pred, blocks=HALVES, fps=FPS):
    segments=[]
    n_transitions=0
    for lo,hi in blocks:
        a=np.asarray(pred[lo:hi+1]); run_start=0
        for t in range(1,len(a)+1):
            if t == len(a) or a[t] != a[run_start]:
                label=a[run_start]
                duration=t-run_start
                if label in ("H","A"):
                    segments.append(duration)
                run_start=t
        n_transitions+=int(np.sum(a[1:]!=a[:-1]))
    return {"nb_possessions_predites":len(segments),
            "duree_moyenne_s":round(float(np.mean(segments))/fps,2) if segments else 0.0,
            "possessions_moins_1s":int(np.sum(np.array(segments)<fps)),
            "nb_transitions_etats":n_transitions,
            "pct_D_sur_frames_notees":round(float(np.mean(np.concatenate([pred[a:b+1] for a,b in blocks]) == "D"))*100,1)}

summary = pd.DataFrame([{"modele":name, **possession_stats(pred)} for name,pred in models.items()])
display(summary)
print("Ne PAS choisir un modèle seulement parce qu'il a le moins de transitions !")

## M4 — 18. Visualiser un petit intervalle

On affiche ici quelques secondes du match :

- en haut, l'écart de distance `d_H - d_A` (quand le ballon est visible) ;
- en bas, les prédictions des différents modèles.

Une bande de prédiction plus régulière peut indiquer un lissage utile **ou excessif** : le graphique est un outil de débogage, pas une évaluation.

In [ ]:
# M4 — Visualiser à partir du premier coup d'envoi, sans créer d'étiquettes manuelles.
VIEW_START = HALVES[0][0]
VIEW_SECONDS = 25
start = VIEW_START
end = min(start+VIEW_SECONDS*FPS, EXPECTED_N)
t = np.arange(start,end)/FPS
fig, (ax1,ax2)=plt.subplots(2,1,figsize=(15,6),sharex=True,gridspec_kw={"height_ratios":[2,2]})
ax1.plot(t,features["nearest_dist_diff"].to_numpy()[start:end],lw=0.9)
ax1.axhline(0,color="gray",ls="--",lw=1)
ax1.set_ylabel("d_H - d_A (m)")
ax1.set_title("Observation de proximité — négatif = joueur H plus proche")
name_list=list(models)
values={"H":0,"A":1,"D":2,"X":3}
for k,name in enumerate(name_list):
    line=np.array([values[v] for v in models[name][start:end]])
    ax2.step(t,line+k*4,where="post",lw=1,label=name)
ax2.set_yticks([k*4 for k in range(len(name_list))],name_list)
ax2.set_xlabel("Temps depuis le début de la vidéo (secondes)")
ax2.set_ylabel("Prédictions")
plt.tight_layout()
plt.show()

## M4 — 19. Générer et valider les CSV de soumission

Format officiel : en-tête `frame,label`, puis **199 319 lignes** avec `frame=0,1,...,199318` ; labels `H/A/D/X`. Ne jamais exporter `frame_id` de l'ATD directement : il commence à **1**, pas à 0.

Tous les modèles sont exportés pour comparaison, mais **un seul fichier final** doit être soumis à la compétition. Pour commencer, le HMM est la version candidate par défaut : ce n'est **pas** une affirmation qu'il est meilleur que le HSMM sur le ground truth secret.

In [ ]:
# Contrôles stricts de format + export de tous les candidats.
def write_and_validate_csv(name, labels):
    path = OUTPUT_DIR / f"Xcelerate_{name}.csv"
    pred = np.asarray(labels, dtype="<U1")
    assert len(pred) == EXPECTED_N
    assert np.isin(pred, ["H","A","D","X"]).all()
    submission=pd.DataFrame({"frame":np.arange(EXPECTED_N,dtype=np.int32),"label":pred})
    submission.to_csv(path,index=False)
    check=pd.read_csv(path)
    assert list(check.columns)==["frame","label"]
    assert len(check)==EXPECTED_N
    assert np.array_equal(check["frame"].to_numpy(),np.arange(EXPECTED_N))
    assert check["label"].isin(["H","A","D","X"]).all()
    return path

exports={name:write_and_validate_csv(name,pred) for name,pred in models.items()}
for name,path in exports.items():
    print(f"{name:16s}: {path}  ({path.stat().st_size/1024/1024:.2f} Mo)")

FINAL_CANDIDATE = "hmm_viterbi"  # changer manuellement si l'équipe retient une autre approche
FINAL_PATH = OUTPUT_DIR / "Xcelerate.csv"
pd.read_csv(exports[FINAL_CANDIDATE]).to_csv(FINAL_PATH,index=False)
print("\nCSV de soumission candidat :",FINAL_PATH)

## M4 — 20. Contrôles supplémentaires et suite du travail d'équipe

**Ce qui fonctionne maintenant :** lecture de tes fichiers Kaggle, extraction des features de M1–M2, récupération correcte des événements M3, quatre familles de modèles (règle brute, vote majoritaire, hystérésis, HMM, HSMM), et génération de CSV valides.

**Ce qui reste à améliorer avant de soumettre :**

1. **Détection de `D`** : les arrêts de jeu ne sont pas déduits correctement par la seule invisibilité du ballon. Demander à M3 des indices précis de reprise / sortie du terrain.
2. **Probabilités d'émission** : les scores actuels sont heuristiques et **non calibrés** ; il faut les discuter physiquement, éventuellement les apprendre de **matchs externes** autorisés.
3. **Durée des états et transitions** : paramètres fixés a priori, non prouvés optimaux ; ne pas utiliser les cinq minutes de GT publique comme jeu de tuning.
4. **Reproductibilité** : conserver les versions des dépendances, les fichiers du code et la commande exacte ; ne jamais committer les données privées du match sur GitHub public.
5. **Évaluation officielle** : utiliser le validateur officiel lorsque le dépôt GitHub public est disponible ; aucune métrique calculée ici n'est un `spellF1` privé.

**IMPORTANT :** le notebook ne télécharge pas les vidéos ni n'exécute d'API cloud. Il analyse uniquement les données que tu as attachées dans l'environnement Kaggle. Pour reproduire exactement les prédictions, utiliser le même dataset et les mêmes versions des bibliothèques.

In [ ]:
# Dernier contrôle synthétique et fichiers utiles à récupérer depuis l'onglet "Output" de Kaggle.
print("Nombre de modèles exportés :",len(exports))
print("Labels du candidat final :")
print(pd.read_csv(FINAL_PATH)["label"].value_counts())
print("\nFichiers prêts :")
print(" -",FEATURES_PATH)
print(" -",FINAL_PATH)
print(" - candidats individuels :", ", ".join(exports))
print("\nFin du notebook. Les résultats doivent encore être validés techniquement par l'équipe.")